# Predicción de Churn

Este notebook extrae la tabla `obt_churn_prediction` desde Snowflake, entrena un modelo RandomForest, y utiliza MLflow para registrar los parámetros y guardar el Pipeline predictivo.

In [12]:
import os
import snowflake.connector
import pandas as pd
import mlflow
import mlflow.sklearn
from dotenv import load_dotenv

# Cargar variables de entorno
load_dotenv()

# Conectar a Snowflake
conn = snowflake.connector.connect(
    user=os.getenv("SNOWFLAKE_USER"),
    password=os.getenv("SNOWFLAKE_PASSWORD"),
    account=os.getenv("SNOWFLAKE_ACCOUNT"),
    warehouse=os.getenv("SNOWFLAKE_WAREHOUSE", "COMPUTE_WH"),
    database=os.getenv("SNOWFLAKE_DATABASE", "HGC_DB"),
    schema="TRAINING_DATASETS"
)

# Extraer los datos
query = "SELECT * FROM training_datasets.obt_churn_prediction"
df = pd.read_sql(query, conn)
df.head()

C:\Users\Robert\AppData\Local\Temp\ipykernel_1804\1062012651.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,ID_CLIENTE_NK,RANGO_EDAD,FEATURE_FRECUENCIA_HISTORICA,FEATURE_MONTO_GASTO_HISTORICO,FEATURE_RECENCIA_DIAS,TARGET_ES_CHURN
0,63680.0,18 - 25,0,0.0,9999,1
1,90046.0,26 - 35,0,0.0,9999,1
2,189531.0,Mayor de 35,0,0.0,9999,1
3,7728.0,Mayor de 35,1,39.0,-146,0
4,33878.0,Mayor de 35,0,0.0,9999,1


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

X = df[['RANGO_EDAD', 'FEATURE_FRECUENCIA_HISTORICA', 'FEATURE_MONTO_GASTO_HISTORICO', 'FEATURE_RECENCIA_DIAS']]
y = df['TARGET_ES_CHURN']

# Dividir dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocesamiento
categorical_features = ['RANGO_EDAD']
numeric_features = ['FEATURE_FRECUENCIA_HISTORICA', 'FEATURE_MONTO_GASTO_HISTORICO', 'FEATURE_RECENCIA_DIAS']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Modelo predictivo (Random Forest)
model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)

# Pipeline final
pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('classifier', model)])

In [14]:
from sklearn.metrics import accuracy_score, classification_report

mlflow.set_experiment("HGC_Churn_Prediction")

with mlflow.start_run():
    # Entrenar pipeline completo
    pipeline.fit(X_train, y_train)
    
    # Predecir y calcular métricas
    y_pred = pipeline.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    
    print(f"Accuracy de validación: {accuracy}")
    print(classification_report(y_test, y_pred))
    
    mlflow.log_metric("accuracy", accuracy)
    
    # MLflow: Registrar el modelo.
    mlflow.sklearn.log_model(
        sk_model=pipeline, 
        artifact_path="churn_model",
        registered_model_name="HGC_Churn_Model"
    )
    
    print("¡Modelo registrado en MLflow con éxito!")

2026/04/14 13:39:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/14 13:39:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Accuracy de validación: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     32804
           1       1.00      1.00      1.00     27196

    accuracy                           1.00     60000
   macro avg       1.00      1.00      1.00     60000
weighted avg       1.00      1.00      1.00     60000

¡Modelo registrado en MLflow con éxito!


Registered model 'HGC_Churn_Model' already exists. Creating a new version of this model...
Created version '2' of model 'HGC_Churn_Model'.


### Para servir este modelo:
Abre tu consola, ubícate en la carpeta (`hgc-ml`) y ejecuta:

`mlflow models serve -m "models:/HGC_Churn_Model/latest" -p 5001 --env-manager local`